# GovIntel QLoRA Fine-Tuning

Kaggle-oriented notebook for Phase 7. It fine-tunes a 4-bit Mistral instruct model on curated GovIntel synthetic JSONL records. Keep generated datasets, adapters, merged models, and notebook outputs out of git.

## Inputs and Secrets

- Training JSONL schema: `{"instruction": ..., "context": ..., "response": ...}`.
- Use a curated 50+ example file created from `python -m govintel.training.synthetic`.
- Store a write-scoped HuggingFace token in Kaggle as `HF_WRITE_TOKEN`; do not reuse the runtime inference token.

In [ ]:
%%capture
!pip install --upgrade --no-cache-dir unsloth bitsandbytes accelerate peft trl transformers datasets huggingface_hub

In [ ]:
import json
import os
from pathlib import Path

import torch
from datasets import Dataset
from huggingface_hub import login
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

DATA_PATH = Path(os.environ.get("GOVINTEL_TRAINING_JSONL", "/kaggle/input/govintel-training/govintel_synthetic_curated.jsonl"))
OUTPUT_DIR = Path("/kaggle/working/govintel-mistral-qlora")
HUB_MODEL_ID = os.environ.get("GOVINTEL_HF_MODEL_ID", "avisheksaha/govintel-mistral-7b")
HF_WRITE_TOKEN = os.environ.get("HF_WRITE_TOKEN", "")

assert DATA_PATH.exists(), f"Missing curated training JSONL: {DATA_PATH}"

In [ ]:
def load_jsonl(path: Path) -> list[dict[str, str]]:
    rows = []
    for line in path.read_text().splitlines():
        row = json.loads(line)
        assert set(row) == {"instruction", "context", "response"}
        assert all(str(row[field]).strip() for field in row)
        rows.append(row)
    assert len(rows) >= 50, "Phase 7 requires at least 50 curated high-quality examples"
    return rows


def format_sft_text(row: dict[str, str]) -> str:
    return (
        "<s>[INST] "
        f"{row['instruction']}\n\nContext:\n{row['context']}"
        " [/INST]\n"
        f"{row['response']}</s>"
    )


records = load_jsonl(DATA_PATH)
held_out_examples = records[-5:]
training_records = records[:-5]
dataset = Dataset.from_list([{"text": format_sft_text(row)} for row in training_records])
len(dataset), len(held_out_examples)

In [ ]:
max_seq_length = 2048
model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
training_args = TrainingArguments(
    output_dir = str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    num_train_epochs = 3,
    learning_rate = 2e-4,
    warmup_ratio = 0.03,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    save_strategy = "epoch",
    report_to = "none",
    seed = 3407,
)

# For a smoke test before a full Kaggle run, temporarily add: max_steps = 5.
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = training_args,
)

trainer.train()

In [ ]:
FastLanguageModel.for_inference(model)

def render_validation_prompt(example: dict[str, str]) -> str:
    return (
        "<s>[INST] "
        f"{example['instruction']}\n\nContext:\n{example['context']}"
        " [/INST]"
    )


validation_outputs = []
for example in held_out_examples[:5]:
    inputs = tokenizer(render_validation_prompt(example), return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=450, temperature=0.0, do_sample=False)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    validation_outputs.append({"expected": example["response"], "generated": decoded})

validation_outputs

In [ ]:
assert HF_WRITE_TOKEN, "Set HF_WRITE_TOKEN as a write-scoped Kaggle secret before publishing"
login(token=HF_WRITE_TOKEN)

adapter_dir = OUTPUT_DIR / "lora_adapters"
merged_dir = OUTPUT_DIR / "merged_16bit"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

model.push_to_hub(str(HUB_MODEL_ID) + "-lora", token=HF_WRITE_TOKEN)
tokenizer.push_to_hub(str(HUB_MODEL_ID) + "-lora", token=HF_WRITE_TOKEN)

# Merge LoRA weights and publish the deployable model for HF Inference.
model.save_pretrained_merged(str(merged_dir), tokenizer, save_method="merged_16bit")
model.push_to_hub_merged(HUB_MODEL_ID, tokenizer, save_method="merged_16bit", token=HF_WRITE_TOKEN)